In [38]:
!ls /kaggle/input

d  finetune2


In [39]:
model_path = "/kaggle/input/finetune2/pytorch/default/1/finetune_best.pt"


In [40]:
import torch

ckpt = torch.load(
    "/kaggle/input/finetune2/pytorch/default/1/finetune_best.pt",
    map_location="cpu"   # đếm param không cần GPU
)

state_dict = ckpt["model_state_dict"]


In [41]:
num_params = sum(v.numel() for v in state_dict.values())
print(f"Total parameters: {num_params:,}")
print(f"In millions: {num_params/1e6:.2f}M")


Total parameters: 90,576,240
In millions: 90.58M


In [42]:
from tokenizers import Tokenizer

tokenizer_src = Tokenizer.from_file("/kaggle/input/d/hongphuc2005/tokenizers/tokenizer_en.json")
tokenizer_tgt = Tokenizer.from_file("/kaggle/input/d/hongphuc2005/tokenizers/tokenizer_vi.json")

sos_idx = tokenizer_tgt.token_to_id("[SOS]")
eos_idx = tokenizer_tgt.token_to_id("[EOS]")
pad_idx = tokenizer_tgt.token_to_id("[PAD]")


In [43]:
print(sos_idx, eos_idx, pad_idx)
print("Vocab EN:", tokenizer_src.get_vocab_size())
print("Vocab VI:", tokenizer_tgt.get_vocab_size())


2 3 1
Vocab EN: 30000
Vocab VI: 30000


In [44]:
import torch

ckpt = torch.load("/kaggle/input/finetune2/pytorch/default/1/finetune_best.pt", map_location="cpu")
state_dict = ckpt["model_state_dict"]

for i, k in enumerate(state_dict.keys()):
    print(k)
    if i == 30:
        break


_orig_mod.encoder.layers.0.self_attention_block.w_q.weight
_orig_mod.encoder.layers.0.self_attention_block.w_q.bias
_orig_mod.encoder.layers.0.self_attention_block.w_k.weight
_orig_mod.encoder.layers.0.self_attention_block.w_k.bias
_orig_mod.encoder.layers.0.self_attention_block.w_v.weight
_orig_mod.encoder.layers.0.self_attention_block.w_v.bias
_orig_mod.encoder.layers.0.self_attention_block.w_o.weight
_orig_mod.encoder.layers.0.self_attention_block.w_o.bias
_orig_mod.encoder.layers.0.feed_forward_block.liner1.weight
_orig_mod.encoder.layers.0.feed_forward_block.liner1.bias
_orig_mod.encoder.layers.0.feed_forward_block.liner2.weight
_orig_mod.encoder.layers.0.feed_forward_block.liner2.bias
_orig_mod.encoder.layers.0.residual_connections.0.norm.alpha
_orig_mod.encoder.layers.0.residual_connections.0.norm.bias
_orig_mod.encoder.layers.0.residual_connections.1.norm.alpha
_orig_mod.encoder.layers.0.residual_connections.1.norm.bias
_orig_mod.encoder.layers.1.self_attention_block.w_q.weight

In [45]:
!ls /kaggle/input/d/hongphuc2005/codetran


config.py  model.py


In [46]:
import sys
sys.path.append("/kaggle/input/d/hongphuc2005/codetran")  # hoặc đường dẫn dataset

from model import build_transformer
from config import get_config
print("ok")


ok


In [47]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [48]:
from config import get_config
config = get_config()
print("ok")

ok


In [49]:
model = build_transformer(
    tokenizer_src.get_vocab_size(),   # src_vocab_size
    tokenizer_tgt.get_vocab_size(),   # tgt_vocab_size
    config["seq_len"],                # src_seq_len
    config["d_model"]                 # d_model
).to(device)


In [50]:
ckpt = torch.load("/kaggle/input/finetune2/pytorch/default/1/finetune_best.pt", map_location="cpu")
state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt


In [51]:
clean_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("_orig_mod."):
        k = k[len("_orig_mod."):]
    clean_state_dict[k] = v


In [52]:
list(clean_state_dict.keys())[:5]


['encoder.layers.0.self_attention_block.w_q.weight',
 'encoder.layers.0.self_attention_block.w_q.bias',
 'encoder.layers.0.self_attention_block.w_k.weight',
 'encoder.layers.0.self_attention_block.w_k.bias',
 'encoder.layers.0.self_attention_block.w_v.weight']

In [53]:
model.load_state_dict(clean_state_dict)
model.eval()


Transformer(
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderBlock(
        (self_attention_block): MultiheadAttentionBlock(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (w_o): Linear(in_features=512, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (feed_forward_block): FeedForwardBlock(
          (liner1): Linear(in_features=512, out_features=2048, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (liner2): Linear(in_features=2048, out_features=512, bias=True)
        )
        (residual_connections): ModuleList(
          (0-1): 2 x ResidualConnection(
            (dropout): Dropout(p=0.1, inplace=False)
            (norm): LayerNormalization()
          )
        )
      )
    )
    (norm): LayerNormalization()
  )
 

In [54]:
from tokenizers import Tokenizer

tokenizer_src = Tokenizer.from_file("/kaggle/input/d/hongphuc2005/tokenizers/tokenizer_en.json")
tokenizer_tgt = Tokenizer.from_file("/kaggle/input/d/hongphuc2005/tokenizers/tokenizer_vi.json")

sos_id = tokenizer_tgt.token_to_id("[SOS]")
eos_id = tokenizer_tgt.token_to_id("[EOS]")
pad_id = tokenizer_tgt.token_to_id("[PAD]")


In [55]:
def causal_mask(size):
    return torch.tril(torch.ones(1, size, size)).bool()


In [56]:
def greedy_translate(sentence, max_len=100):
    model.eval()
    device = next(model.parameters()).device

    # encode source
    src_ids = tokenizer_src.encode(sentence).ids
    src = torch.tensor(src_ids).unsqueeze(0).to(device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool).to(device)

    encoder_output = model.encode(src, src_mask)

    # decoder start with SOS
    tgt = torch.tensor([[sos_id]], device=device)

    for _ in range(max_len):
        tgt_mask = causal_mask(tgt.size(1)).to(device)

        out = model.decode(encoder_output, src_mask, tgt, tgt_mask)
        probs = model.project(out[:, -1])
        next_token = torch.argmax(probs, dim=-1).item()

        if next_token == eos_id:
            break

        tgt = torch.cat([tgt, torch.tensor([[next_token]], device=device)], dim=1)

    return tokenizer_tgt.decode(tgt.squeeze(0).tolist())


In [57]:
greedy_translate("i go to school.", max_len=10)


'i đến trường học .'

In [58]:
import torch

def beam_search_translate(
    model,
    sentence,
    tokenizer_src,
    tokenizer_tgt,
    beam_size=4,
    max_len=100
):
    model.eval()
    device = next(model.parameters()).device

    sos = tokenizer_tgt.token_to_id("[SOS]")
    eos = tokenizer_tgt.token_to_id("[EOS]")

    # -------- Encode source --------
    src_ids = tokenizer_src.encode(sentence).ids
    src = torch.tensor(src_ids).unsqueeze(0).to(device)
    src_mask = torch.ones(1, 1, src.size(1), dtype=torch.bool).to(device)

    with torch.no_grad():
        encoder_output = model.encode(src, src_mask)

    # beam = (tokens, log_prob)
    beams = [([sos], 0.0)]

    for _ in range(max_len):
        new_beams = []

        for tokens, score in beams:
            # nếu đã kết thúc thì giữ nguyên
            if tokens[-1] == eos:
                new_beams.append((tokens, score))
                continue

            tgt = torch.tensor(tokens).unsqueeze(0).to(device)
            tgt_mask = torch.tril(
                torch.ones(1, tgt.size(1), tgt.size(1), device=device)
            ).bool()

            with torch.no_grad():
                out = model.decode(encoder_output, src_mask, tgt, tgt_mask)
                log_probs = model.project(out[:, -1]).squeeze(0)

            topk = torch.topk(log_probs, beam_size)

            for i in range(beam_size):
                next_token = topk.indices[i].item()
                next_score = score + topk.values[i].item()
                new_beams.append((tokens + [next_token], next_score))

        # nếu beam chết thì dừng
        if len(new_beams) == 0:
            break

        # giữ top beam
        beams = sorted(new_beams, key=lambda x: x[1] / len(x[0]), reverse=True)
        beams = beams[:beam_size]

        # stop sớm nếu tất cả đều EOS
        if all(b[0][-1] == eos for b in beams):
            break

    if len(beams) == 0:
        return ""

    best_tokens = beams[0][0]

    # bỏ SOS và EOS
    return tokenizer_tgt.decode(best_tokens[1:-1])


In [61]:
print(
    beam_search_translate(
        model,
        "I love you .",
        tokenizer_src,
        tokenizer_tgt,
        beam_size=4
    )
)


Tôi yêu thích bạn .
